In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print("Loading datasets...")

race_df = pd.read_csv('data/processed/race_results_2025-6_clean.csv')
reddit_df = pd.read_csv('data/raw/reddit_posts.csv')
youtube_df = pd.read_csv('data/raw/youtube_videos_sponsor_2025-6.csv')
news_df = pd.read_csv('data/raw/news_mentions_2025-6_raw.csv')

print(f"Race Results: {len(race_df)} rows")
print(f"Reddit Mentions: {len(reddit_df)} rows")
print(f"YouTube Engagement: {len(youtube_df)} rows")
print(f"News Mentions: {len(news_df)} rows")

Loading datasets...
Race Results: 2013 rows
Reddit Mentions: 1884 rows
YouTube Engagement: 1944 rows
News Mentions: 57 rows


In [2]:
def audit_dataset(df, name):
    """
    Print comprehensive audit of dataset structure.
    """
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumn Names and Types:")
    print("-" * 40)
    for col in df.columns:
        dtype = df[col].dtype
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
        print(f"  {col:25} | {str(dtype):10} | Sample: {str(sample)[:30]}")
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  None")
    return None

# Audit each dataset
audit_dataset(race_df, "Race Results")
audit_dataset(reddit_df, "Reddit Mentions")
audit_dataset(youtube_df, "YouTube Engagement")
audit_dataset(news_df, "News Mentions")


DATASET: Race Results
Shape: 2013 rows x 33 columns

Column Names and Types:
----------------------------------------
  year                      | int64      | Sample: 2025
  race_number               | int64      | Sample: 1
  race_name                 | str        | Sample: 2025 Daytona 500
  Race_Date                 | str        | Sample: 2025-02-16
  track                     | str        | Sample: Daytona International Speedway
  track_type                | str        | Sample: road course
  track_miles               | float64    | Sample: 2.4
  total_laps                | float64    | Sample: 95.0
  caution_flags             | int64      | Sample: 8
  caution_laps              | int64      | Sample: 47
  lead_changes              | int64      | Sample: 56
  avg_speed_mph             | float64    | Sample: 129.159
  pole_speed_mph            | float64    | Sample: 182.745
  margin_of_victory         | str        | Sample: .113 sec
  attendance                | float64    | Samp

In [3]:
# Define standard date format
DATE_FORMAT = '%Y-%m-%d'  # ISO 8601: 2024-02-18

def standardize_dates(df, date_columns, parse_format=None):
    """
    Convert all date columns to standard format.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    date_columns : list
        List of column names containing dates
    parse_format : str, optional
        Explicit format to use when parsing the input column (e.g. '%Y%m%d%H%M%S').
        Needed for numeric-looking date columns like 'seendate', since pandas
        otherwise treats an int64 column as nanoseconds since epoch instead of
        parsing it as a date string.

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized dates
    """
    df = df.copy()

    for col in date_columns:
        if col not in df.columns:
            print(f"  Warning: Column '{col}' not found")
            continue

        # Convert to datetime
        if parse_format:
            df[col] = pd.to_datetime(df[col].astype(str), format=parse_format, errors='coerce')
        else:
            df[col] = pd.to_datetime(df[col], errors='coerce')

        # Check for conversion failures
        failed = df[col].isna().sum()
        if failed > 0:
            print(f"  Warning: {failed} dates failed to convert in '{col}'")

        # Convert to standard string format for CSV compatibility
        df[f'{col}_str'] = df[col].dt.strftime(DATE_FORMAT)

    return df

# Apply to each dataset
print("Standardizing dates...")

race_df = standardize_dates(race_df, ['Race_Date'])
reddit_df = standardize_dates(reddit_df, ['race_date'] if 'race_date' in reddit_df.columns else [])
youtube_df = standardize_dates(youtube_df, ['race_date'])

news_df = standardize_dates(news_df, ['seendate'], parse_format='%Y%m%d%H%M%S')

print("Date standardization complete.")

Standardizing dates...
Date standardization complete.


In [4]:
def standardize_columns(df, column_mapping):
    """
    Rename columns to standard names and convert to lowercase with underscores.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    column_mapping : dict
        Dictionary mapping old names to new names

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized column names
    """
    df = df.copy()

    # Apply explicit mappings
    df = df.rename(columns=column_mapping)

    # Convert remaining columns to lowercase with underscores
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    return df

# Define standard column names for each dataset
race_columns = {
    'Race_Name': 'race_name',
    'Race_Date': 'race_date',
    'Race_Number': 'race_number',
    'Driver': 'driver',
    'Team': 'team',
    'Sponsor': 'sponsor',
    'Finish_Position': 'finish_position',
    'Laps_Led': 'laps_led'
}

exposure_columns = {
    'race_period': 'race_name',  # If Reddit uses race_period
    'Race_Name': 'race_name',
    'Race_Number': 'race_number',
    'Sponsor': 'sponsor'
}

# Apply standardization
race_df = standardize_columns(race_df, race_columns)
reddit_df = standardize_columns(reddit_df, exposure_columns)
youtube_df = standardize_columns(youtube_df, exposure_columns)
news_df = standardize_columns(news_df, exposure_columns)

print("Column name standardization complete.")

Column name standardization complete.


In [5]:
# add race numbers based on date:
date_to_number = {pd.Timestamp('2025-02-1'): 1,
                  pd.Timestamp('2025-2-12'): 2,
                  pd.Timestamp('2025-2-22'): 3,
                  pd.Timestamp('2025-2-28'): 4,
                  pd.Timestamp('2025-3-8'): 5,
                  pd.Timestamp('2025-3-15'): 6,
                  pd.Timestamp('2025-3-22'): 7,
                  pd.Timestamp('2025-3-30'): 8,
                  pd.Timestamp('2025-4-5'): 9,
                  pd.Timestamp('2025-4-12'): 10,
                  pd.Timestamp('2025-4-26'): 11,
                  pd.Timestamp('2025-5-3'): 12,
                  pd.Timestamp('2025-5-10'): 13,
                  pd.Timestamp('2025-5-17'): 14,
                  pd.Timestamp('2025-5-24'): 15,
                  pd.Timestamp('2025-5-31'): 16,
                  pd.Timestamp('2025-6-7'): 17,
                  pd.Timestamp('2025-6-14'): 18,
                  pd.Timestamp('2025-6-21'): 19,
                  pd.Timestamp('2025-6-27'): 20,
                  pd.Timestamp('2025-7-5'): 21,
                  pd.Timestamp('2025-7-12'): 22,
                  pd.Timestamp('2025-7-19'): 23,
                  pd.Timestamp('2025-7-26'): 24,
                  pd.Timestamp('2025-8-2'): 25,
                  pd.Timestamp('2025-8-9'): 26,
                  pd.Timestamp('2025-8-15'): 27,
                  pd.Timestamp('2025-8-22'): 28,
                  pd.Timestamp('2025-8-30'): 29,
                  pd.Timestamp('2025-9-6'): 30,
                  pd.Timestamp('2025-9-12'): 31,
                  pd.Timestamp('2025-9-20'): 32,
                  pd.Timestamp('2025-9-27'): 33,
                  pd.Timestamp('2025-10-4'): 34,
                  pd.Timestamp('2025-10-12'): 35,
                  pd.Timestamp('2025-10-18'): 36,
                  pd.Timestamp('2025-10-27'): 37,
                  pd.Timestamp('2025-11-1'): 38,
                  pd.Timestamp('2026-2-21'): 39,
                  pd.Timestamp('2026-2-28'): 40,
                  pd.Timestamp('2026-3-7'): 41,
                  pd.Timestamp('2026-3-14'): 42,
                  pd.Timestamp('2026-3-21'): 43,
                  pd.Timestamp('2026-3-28'): 44,
                  pd.Timestamp('2026-4-11'): 45,
                  pd.Timestamp('2026-4-18'): 46,
                  pd.Timestamp('2026-4-25'): 47,
                  pd.Timestamp('2026-5-2'): 48,
                  pd.Timestamp('2026-5-9'): 49,
                  pd.Timestamp('2026-5-16'): 50,
                  pd.Timestamp('2026-5-23'): 51,
                  pd.Timestamp('2026-5-30'): 52,
                  pd.Timestamp('2026-6-6'): 53,
}
race_series = pd.Series(date_to_number).sort_index()

def assign_race_number(df, date_col='published')-> pd.DataFrame:
    """Assign race numbers based on the most recent race date on or before the given date column."""
    df = df.copy()
    normalized = pd.to_datetime(df[date_col], utc=True).dt.tz_localize(None).dt.normalize()
    df['race_number'] = normalized.apply(lambda d: race_series.asof(d))
    return df
reddit_df = assign_race_number(reddit_df, 'published')
news_df = assign_race_number(news_df, 'seendate')

In [6]:
race_num_to_name = (race_df[['race_number', 'race_name']].drop_duplicates().set_index('race_number')['race_name']).to_dict()

def assign_race_name(df, race_num_col='race_number')-> pd.DataFrame:
    """Assign race names based on race numbers."""
    df = df.copy()
    df['race_name'] = df[race_num_col].map(race_num_to_name)
    return df

reddit_df = assign_race_name(reddit_df, 'race_number')
news_df = assign_race_name(news_df, 'race_number')
youtube_df = assign_race_name(youtube_df, 'race_number')

In [7]:
def create_race_name_mapping(df, race_col='race_name'):
    """
    Create a standardized mapping for race names.
    Returns dict mapping original names to standardized names.
    """
    unique_races = df[race_col].unique()
    print(f"Found {len(unique_races)} unique race names:")
    for race in sorted(unique_races):
        print(f"  - {race}")
    return unique_races

# Check race names in each dataset
print("\n=== Race Names in Race Results ===")
race_names_main = create_race_name_mapping(race_df)

print("\n=== Race Names in Reddit Data ===")
race_names_reddit = create_race_name_mapping(reddit_df)

print("\n=== Race Names in YouTube Data ===")
race_names_youtube = create_race_name_mapping(youtube_df)

print("\n=== Race Names in News Data ===")
race_names_news = create_race_name_mapping(news_df)


=== Race Names in Race Results ===
Found 52 unique race names:
  - 2025 AdventHealth 400
  - 2025 Ambetter Health 400
  - 2025 Autotrader EchoPark Automotive 400
  - 2025 Bank of America ROVAL 400
  - 2025 Bass Pro Shops Night Race
  - 2025 Brickyard 400 Presented by PPG
  - 2025 Coca-Cola 600
  - 2025 Coke Zero Sugar 400
  - 2025 Cook Out 400
  - 2025 Cracker Barrel 400
  - 2025 Cup Series Championship
  - 2025 Daytona 500
  - 2025 EchoPark Automotive Grand Prix
  - 2025 Enjoy Illinois 300
  - 2025 Firekeepers Casino 400
  - 2025 Food City 500
  - 2025 Go Bowling at The Glen
  - 2025 Goodyear 400
  - 2025 Grant Park 165
  - 2025 Hollywood Casino 400
  - 2025 Iowa Corn 350
  - 2025 Jack Links 500
  - 2025 Mobil 1 301
  - 2025 Pennzoil 400
  - 2025 Quaker State 400 available at Walmart
  - 2025 Shriners Childrens 500
  - 2025 South Point 400
  - 2025 Southern 500
  - 2025 Straight Talk Wireless 400
  - 2025 The Great American Getaway 400
  - 2025 Toyota / Save Mart 350
  - 2025 Viva Me

In [8]:
# Create master race name mapping
# This ensures all datasets use exactly the same race names

RACE_NAME_MAPPING = {
    # Add variations you find in your data
    '2025 Daytona 500': '2025 Daytona 500',
    '2025 DAYTONA 500': '2025 Daytona 500',
    '2025 Ambetter Health 400': '2025 Ambetter Health 400',
    '2025 Ambetter 400': '2025 Ambetter Health 400',
    '2025 Pennzoil 400': '2025 Pennzoil 400',
    '2025 Pennzoil 400 Las Vegas': '2025 Pennzoil 400',
    '2025 Wurth 400 presented by LIQUI MOLLY': '2025 Wurth 400',
    '2025 Quaker State 400 available at Walmart': '2025 Quaker State 400',
    '2025 Toyota / Save Mart 350': '2025 Toyota Save Mart 350',
    '2025 Brickyard 400 Presented by PPG': '2025 Brickyard 400',
    '2026 Autotrader 400 at EchoPark Speedway': '2026 Autotrader 400',
    '2026 DuraMax Texas Grand Prix Presented by RelaDyne': '2026 DuraMax Texas Grand Prix',
    '2026 Pennzoil 400 Presented by Jiffy Lube': '2026 Pennzoil 400',
    '2026 Wurth 400 Presented by LIQUI MOLY': '2026 Wurth 400',
    
}

def standardize_race_names(df, mapping, race_col='race_name'):
    """
    Apply race name standardization.
    """
    df = df.copy()

    # Apply mapping where available
    df[race_col] = df[race_col].replace(mapping)

    # Check for unmapped races
    unmapped = df[~df[race_col].isin(mapping.values())][race_col].unique()
    if len(unmapped) > 0:
        print(f"Warning: {len(unmapped)} race names not in mapping:")
        for name in unmapped:
            print(f"  - '{name}'")

    return df

# Apply to all datasets
race_df = standardize_race_names(race_df, RACE_NAME_MAPPING)
reddit_df = standardize_race_names(reddit_df, RACE_NAME_MAPPING)
youtube_df = standardize_race_names(youtube_df, RACE_NAME_MAPPING)
news_df = standardize_race_names(news_df, RACE_NAME_MAPPING)

  - '2025 EchoPark Automotive Grand Prix'
  - '2025 Shriners Childrens 500'
  - '2025 Straight Talk Wireless 400'
  - '2025 Cook Out 400'
  - '2025 Goodyear 400'
  - '2025 Food City 500'
  - '2025 Jack Links 500'
  - '2025 Wurth 400 Presented by LIQUI MOLY'
  - '2025 AdventHealth 400'
  - '2025 Coca-Cola 600'
  - '2025 Cracker Barrel 400'
  - '2025 Firekeepers Casino 400'
  - '2025 Viva Mexico 250'
  - '2025 The Great American Getaway 400'
  - '2025 Grant Park 165'
  - '2025 Autotrader EchoPark Automotive 400'
  - '2025 Iowa Corn 350'
  - '2025 Go Bowling at The Glen'
  - '2025 Coke Zero Sugar 400'
  - '2025 Southern 500'
  - '2025 Enjoy Illinois 300'
  - '2025 Bass Pro Shops Night Race'
  - '2025 Mobil 1 301'
  - '2025 Hollywood Casino 400'
  - '2025 Bank of America ROVAL 400'
  - '2025 South Point 400'
  - '2025 Yellawood 500'
  - '2025 Xfinity 500'
  - '2025 Cup Series Championship'
  - '2026 Daytona 500'
  - '2026 Straight Talk Wireless 500'
  - '2026 Goodyear 400'
  - '2026 Cook O

In [9]:
# Sponsor name standardization
SPONSOR_NAME_MAPPING = {
    'FedEx': 'FedEx',
    'Fedex': 'FedEx',
    'FEDEX': 'FedEx',
    'NAPA': 'NAPA',
    'NAPA Auto Parts': 'NAPA',
    'Napa': 'NAPA',
    "McDonald's": "McDonald's",
    'McDonalds': "McDonald's",
    "Mcdonald's": "McDonald's",
    'Ally': 'Ally',
    'Ally Financial': 'Ally',
    'Ally Racing': 'Ally',
    'Busch Light': 'Busch Light',
    'Busch': 'Busch Light',
    'Busch Light Apple': 'Busch Light',
    'progressive': 'Progressive',
    'Progressive Insurance': 'Progressive',
    'progressive insurance': 'Progressive',
    'Progressive insurance': 'Progressive',
    'castrol': 'Castrol',
    'Love\'s Travel Stops': 'Love\'s Travel Stops',
    'Loves Travel Stops': 'Love\'s Travel Stops',
    'love\'s travel stops': 'Love\'s Travel Stops',
    'Love\'s': 'Love\'s Travel Stops',
    'Love’s Travel Stops': 'Love\'s Travel Stops',
    'love’s travel stops': 'Love\'s Travel Stops',
    'Love’s': 'Love\'s Travel Stops',
    'love’s': 'Love\'s Travel Stops',
    'Cheddar\'s Scratch Kitchen': 'Cheddar\'s Scratch Kitchen',
    'Cheddar’s Scratch Kitchen': 'Cheddar\'s Scratch Kitchen',
    'Cheddar’s': 'Cheddar\'s Scratch Kitchen',
    'Cheddar\'s': 'Cheddar\'s Scratch Kitchen',
    'cheddar’s': 'Cheddar\'s Scratch Kitchen',
    'cheddar\'s': 'Cheddar\'s Scratch Kitchen',
    'cheddar’s scratch kitchen': 'Cheddar\'s Scratch Kitchen',
    'cheddar\'s scratch kitchen': 'Cheddar\'s Scratch Kitchen',
}

def standardize_sponsor_names(df, mapping, sponsor_col='sponsor'):
    """
    Apply sponsor name standardization.
    """
    df = df.copy()
    df[sponsor_col] = df[sponsor_col].replace(mapping)

    # Verify all sponsors are standardized
    target_sponsors = ['Progressive', 'Castrol', 'Cheddar\'s Scratch Kitchen', 'Love\'s Travel Stops', 'Busch Light']
    unique_sponsors = df[sponsor_col].unique()
    unexpected = [s for s in unique_sponsors if s not in target_sponsors]
    if unexpected:
        print(f"Warning: Unexpected sponsors found: {unexpected}")

    return df

# Apply to all datasets
reddit_df = standardize_sponsor_names(reddit_df, SPONSOR_NAME_MAPPING)
news_df = standardize_sponsor_names(news_df, SPONSOR_NAME_MAPPING)


In [10]:
def consolidate_sponsor_mentions(df, sponsor_names, prefix='features_', new_col='sponsor'):
    """
    Collapse the per-sponsor boolean mention columns (e.g. 'features_progressive',
    'features_castrol') into a single column holding the sponsor name(s) mentioned
    in that row.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing one boolean column per sponsor
    sponsor_names : list
        Canonical sponsor names (e.g. from SPONSOR_NAME_MAPPING), used to recover
        proper casing/punctuation lost when column names were lowercased
    prefix : str
        Prefix shared by all the mention columns
    new_col : str
        Name of the resulting sponsor column

    Returns:
    --------
    pd.DataFrame : DataFrame with the mention columns replaced by `new_col`
    """
    df = df.copy()

    mention_cols = [c for c in df.columns if c.startswith(prefix)]
    if not mention_cols:
        print(f"Warning: no columns found with prefix '{prefix}'")
        return df

    # Map each column's normalized suffix back to its canonical sponsor name
    normalized_to_canonical = {name.lower().replace(' ', '_'): name for name in sponsor_names}
    col_to_sponsor = {col: normalized_to_canonical.get(col[len(prefix):], col[len(prefix):]) for col in mention_cols}

    def sponsors_in_row(row):
        mentioned = [col_to_sponsor[c] for c in mention_cols if row[c]]
        return ', '.join(mentioned) if mentioned else None

    df[new_col] = df[mention_cols].apply(sponsors_in_row, axis=1)
    df = df.drop(columns=mention_cols)

    return df

# Collapse features_* columns into a single 'sponsor' column
youtube_df = consolidate_sponsor_mentions(youtube_df, sponsor_names=list(SPONSOR_NAME_MAPPING.values()))
print("Sponsors found in YouTube data:")
print(youtube_df['sponsor'].value_counts(dropna=False))

Sponsors found in YouTube data:
sponsor
NaN                                 942
Cheddar's Scratch Kitchen           259
Progressive                         244
Love's Travel Stops                 229
Busch Light                         197
Castrol                              70
Progressive, Busch Light              2
Progressive, Love's Travel Stops      1
Name: count, dtype: int64


In [11]:
def create_merge_key(df, components=['race_number', 'sponsor']):
    """
    Create a unique merge key from multiple columns.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to add merge key to
    components : list
        Columns to combine into merge key

    Returns:
    --------
    pd.DataFrame : DataFrame with merge_key column added
    """
    df = df.copy()

    # Ensure components exist
    missing = [c for c in components if c not in df.columns]
    if missing:
        print(f"Warning: Columns missing for merge key: {missing}")
        return df

    # Create merge key
    df['merge_key'] = df[components].fillna('unknown').astype(str).agg('_'.join, axis=1)

    return df

# Add merge keys to all datasets
race_agg = race_df.groupby(['race_number', 'primary_sponsor']).first().reset_index()
race_agg = create_merge_key(race_agg, components=['race_number', 'primary_sponsor'])

reddit_df = create_merge_key(reddit_df)
youtube_df = create_merge_key(youtube_df)
news_df = create_merge_key(news_df)

print("Sample merge keys:")
print(race_agg['merge_key'].head(15).tolist())

Sample merge keys:
['1_Busch Light', '1_Castrol', "1_Love's Travel Stops", '1_Other/Unknown', '1_Progressive', '2_Busch Light', '2_Castrol', "2_Love's Travel Stops", '2_Other/Unknown', '2_Progressive', '3_Busch Light', '3_Castrol', "3_Love's Travel Stops", '3_Other/Unknown', '3_Progressive']


In [12]:
# Save standardized versions
race_df.to_csv('data/processed/race_results_standardized.csv', index=False)
reddit_df.to_csv('data/processed/reddit_mentions_standardized.csv', index=False)
youtube_df.to_csv('data/processed/youtube_engagement_standardized.csv', index=False)
news_df.to_csv('data/processed/news_mentions_standardized.csv', index=False)

print("Standardized datasets saved.")

# Document the standardization rules
standardization_log = {
    'date_format': DATE_FORMAT,
    'race_name_mapping_count': len(RACE_NAME_MAPPING),
    'sponsor_name_mapping_count': len(SPONSOR_NAME_MAPPING),
    'merge_key_components': ['race_number', 'sponsor'],
    'standardization_date': datetime.now().isoformat()
}

import json
with open('data/processed/standardization_log.json', 'w') as f:
    json.dump(standardization_log, f, indent=2)

print("Standardization log saved.")

Standardized datasets saved.
Standardization log saved.


In [13]:
# Load standardized data
race_df = pd.read_csv('data/processed/race_results_standardized.csv')
reddit_df = pd.read_csv('data/processed/reddit_mentions_standardized.csv')
youtube_df = pd.read_csv('data/processed/youtube_engagement_standardized.csv')
news_df = pd.read_csv('data/processed/news_mentions_standardized.csv')

# Create race-level summary from race results
# (Each row = one sponsor at one race)
race_base = race_df.groupby(['race_number', 'race_name', 'race_date', 'primary_sponsor']).agg({
    'finish_position': 'min',  # Best finish among cars carrying that sponsor
    'laps_led': 'sum'  # Total laps led by cars carrying that sponsor
}).reset_index()

# Add merge key
race_base['merge_key'] = race_base['race_number'].astype(str) + '_' + race_base['primary_sponsor']

n_races = race_df['race_number'].nunique()
n_sponsors = race_df['primary_sponsor'].nunique()
print(f"Base dataset: {len(race_base)} sponsor-race combinations")
print(f"Expected: {n_races} races x {n_sponsors} sponsors = {n_races * n_sponsors} rows")
print(f"Actual: {len(race_base)} rows")

# Check for any sponsor-race combinations
print("\nSponsor coverage:")
print(race_base.groupby('primary_sponsor').size())

Base dataset: 276 sponsor-race combinations
Expected: 53 races x 6 sponsors = 318 rows
Actual: 276 rows

Sponsor coverage:
primary_sponsor
Busch Light                  53
Castrol                      53
Love's Travel Stops          53
Other/Unknown                53
Progressive                  52
cheddar's Scratch Kitchen    12
dtype: int64


In [14]:
# all I need is the number of reddit posts per sponsor per race, I wasnt able to get engagement data
def sum_engagement_per_race(df:pd.DataFrame)->pd.DataFrame:
    num_mentions = df[['merge_key']].value_counts()
    # merge key is sponsor and race number in one
    print(num_mentions.head(10))
    return num_mentions
reddit_mentions = sum_engagement_per_race(reddit_df)
reddit_mentions_df = reddit_mentions.reset_index()
reddit_mentions_df = reddit_mentions_df.rename(columns={'count':'reddit_mention_count'})

# Perform left join
master_df = race_base.merge(
    reddit_mentions_df,
    on='merge_key',
    how='left',
    indicator='_reddit_merge'
)

# Check merge results
print("\nReddit merge results:")
print(master_df['_reddit_merge'].value_counts())

# Count matches
matched = (master_df['_reddit_merge'] == 'both').sum()
total = len(master_df)
print(f"Match rate: {matched}/{total} ({matched/total*100:.1f}%)")

# Unmatched sponsor-races had zero reddit posts, not a missing value
master_df['reddit_mention_count'] = master_df['reddit_mention_count'].fillna(0)
master_df = master_df.drop(columns=['_reddit_merge'])

merge_key                   
38_Progressive                  156
38_Castrol                       77
38_Busch Light                   60
53_Progressive                   45
38_Love's Travel Stops           41
6_Progressive                    33
38_Cheddar's Scratch Kitchen     30
53_Cheddar's Scratch Kitchen     26
24_Cheddar's Scratch Kitchen     22
26_Cheddar's Scratch Kitchen     18
Name: count, dtype: int64

Reddit merge results:
_reddit_merge
both          211
left_only      65
right_only      0
Name: count, dtype: int64
Match rate: 211/276 (76.4%)


In [15]:
# Prepare YouTube data for merge
youtube_merge = youtube_df[['merge_key', 'view_count', 'like_count','comment_count','duration']].copy()

# duration comes from the YouTube API as an ISO 8601 duration string (e.g. 'PT5M50S'),
# so it has to be converted to seconds before it can be summed
youtube_merge['duration_seconds'] = pd.to_timedelta(youtube_merge['duration']).dt.total_seconds()

youtube_agg = youtube_merge.groupby('merge_key').agg(
    youtube_total_view_count=('view_count', 'sum'),
    youtube_total_like_count=('like_count', 'sum'),
    youtube_total_comment_count=('comment_count', 'sum'),
    youtube_total_duration_seconds=('duration_seconds', 'sum'),
    num_youtube_videos=('merge_key', 'count')
).reset_index()


# Check for duplicate merge keys
dupes = youtube_agg['merge_key'].duplicated().sum()
if dupes > 0:
    print(f"Warning: {dupes} duplicate merge keys in YouTube data")
    youtube_merge = youtube_merge.drop_duplicates(subset='merge_key', keep='first')

# Perform left join
master_df = master_df.merge(
    youtube_agg,
    on='merge_key',
    how='left',
    indicator='_youtube_merge'
)

# Check merge results
print("\nYouTube merge results:")
print(master_df['_youtube_merge'].value_counts())

matched = (master_df['_youtube_merge'] == 'both').sum()
print(f"Match rate: {matched}/{len(master_df)} ({matched/len(master_df)*100:.1f}%)")

# Handle unmatched
youtube_cols = ['youtube_total_view_count', 'num_youtube_videos',
                'youtube_total_like_count', 'youtube_total_comment_count', 'youtube_total_duration_seconds']
for col in youtube_cols:
    master_df[col] = master_df[col].fillna(0)

master_df = master_df.drop(columns=['_youtube_merge'])


YouTube merge results:
_youtube_merge
left_only     144
both          132
right_only      0
Name: count, dtype: int64
Match rate: 132/276 (47.8%)


In [16]:
# Prepare News data for merge
news_merge = news_df[['merge_key', 'mention_type',
                       'tier_weight']].copy()

news_agg = news_df.groupby('merge_key').agg(
    news_total_mentions=('merge_key','count'),
    news_weighted_mentions=('tier_weight','sum')
).reset_index()

mention_type_counts = pd.crosstab(news_df['merge_key'], news_df['mention_type']).reset_index()
mention_type_counts = mention_type_counts.rename(columns={
    'primary': 'news_primary_mentions',
    'secondary': 'news_secondary_mentions',
    'passing': 'news_passing_mentions'
})

news_agg = news_agg.merge(mention_type_counts, on='merge_key', how='left')

# Check for duplicate merge keys
dupes = news_agg['merge_key'].duplicated().sum()
if dupes > 0:
    print(f"Warning: {dupes} duplicate merge keys in News data")
    news_merge = news_merge.drop_duplicates(subset='merge_key', keep='first')

# Perform left join
master_df = master_df.merge(
    news_agg,
    on='merge_key',
    how='left',
    indicator='_news_merge'
)

# Check merge results
print("\nNews merge results:")
print(master_df['_news_merge'].value_counts())

matched = (master_df['_news_merge'] == 'both').sum()
print(f"Match rate: {matched}/{len(master_df)} ({matched/len(master_df)*100:.1f}%)")

# Handle unmatched
news_cols = ['news_total_mentions', 'news_weighted_mentions', 'news_primary_mentions','news_secondary_mentions','news_passing_mentions']
for col in news_cols:
    master_df[col] = master_df[col].fillna(0)

master_df = master_df.drop(columns=['_news_merge'])


News merge results:
_news_merge
left_only     241
both           35
right_only      0
Name: count, dtype: int64
Match rate: 35/276 (12.7%)


In [ ]:
def spot_check_merge(master_df, race_number, sponsor, original_dfs):
    """
    Verify merge accuracy by recomputing each aggregate directly from the
    original per-row source data and comparing it to the merged value.

    Parameters:
    -----------
    master_df : pd.DataFrame
        Merged master dataset
    race_number : int
        Race number to check
    sponsor : str
        Sponsor name to check
    original_dfs : dict
        Dictionary of original DataFrames {'reddit': df, 'youtube': df, 'news': df}
    """
    print(f"\n{'='*60}")
    print(f"SPOT CHECK: Race {race_number} - {sponsor}")
    print(f"{'='*60}")

    # Get merged record
    merged = master_df[
        (master_df['race_number'] == race_number) &
        (master_df['primary_sponsor'] == sponsor)
    ]

    if len(merged) == 0:
        print("No matching record found in master dataset!")
        return

    merged = merged.iloc[0]
    merge_key = merged['merge_key']

    print(f"\nMerge Key: {merge_key}")
    print(f"Race Name: {merged['race_name']}")
    print(f"Finish Position: {merged['finish_position']}")

    # Check Reddit values
    print(f"\n--- Reddit Data ---")
    print(f"Master: {merged['reddit_mention_count']:.0f} mentions")

    reddit_original = original_dfs['reddit'][original_dfs['reddit']['merge_key'] == merge_key]
    original_reddit_count = len(reddit_original)
    print(f"Original: {original_reddit_count} posts found")
    print("MATCH" if merged['reddit_mention_count'] == original_reddit_count else "MISMATCH!")

    # Check YouTube values
    print(f"\n--- YouTube Data ---")
    print(f"Master: {merged['youtube_total_view_count']:,.0f} total views across {merged['num_youtube_videos']:.0f} videos")

    youtube_original = original_dfs['youtube'][original_dfs['youtube']['merge_key'] == merge_key]
    original_view_count = youtube_original['view_count'].sum()
    print(f"Original: {original_view_count:,.0f} total views across {len(youtube_original)} videos")
    print("MATCH" if merged['youtube_total_view_count'] == original_view_count else "MISMATCH!")

    # Check News values
    print(f"\n--- News Data ---")
    print(f"Master: {merged['news_total_mentions']:.0f} mentions")

    news_original = original_dfs['news'][original_dfs['news']['merge_key'] == merge_key]
    original_news_count = len(news_original)
    print(f"Original: {original_news_count} mentions found")
    print("MATCH" if merged['news_total_mentions'] == original_news_count else "MISMATCH!")

# Run spot checks on several records
original_dfs = {
    'reddit': reddit_df,
    'youtube': youtube_df,
    'news': news_df
}

# Check high-profile race
spot_check_merge(master_df, 1, 'Progressive', original_dfs)

# Check mid-season race
spot_check_merge(master_df, 18, 'Castrol', original_dfs)

# Check late-season race
spot_check_merge(master_df, 35, "Busch Light", original_dfs)


SPOT CHECK: Race 1 - Progressive

Merge Key: 1_Progressive
Race Name: 2025 Daytona 500
Finish Position: 24

--- Reddit Data ---
Master: 12 mentions
Original: 12 posts found
MATCH

--- YouTube Data ---
Master: 172,641 total views across 5 videos
Original: 172,641 total views across 5 videos
MATCH

--- News Data ---
Master: 0 mentions
Original: 0 mentions found
MATCH

SPOT CHECK: Race 18 - Castrol

Merge Key: 18_Castrol
Race Name: 2025 Quaker State 400
Finish Position: 2

--- Reddit Data ---
Master: 3 mentions
Original: 3 posts found
MATCH

--- YouTube Data ---
Master: 131,971 total views across 2 videos
Original: 131,971 total views across 2 videos
MATCH

--- News Data ---
Master: 0 mentions
Original: 0 mentions found
MATCH

SPOT CHECK: Race 35 - Busch Light

Merge Key: 35_Busch Light
Race Name: 2025 Xfinity 500
Finish Position: 4

--- Reddit Data ---
Master: 1 mentions
Original: 1 posts found
MATCH

--- YouTube Data ---
Master: 0 total views across 0 videos
Original: 0 total views acr

In [19]:
print("\n" + "="*60)
print("MASTER DATASET SUMMARY")
print("="*60)

print(f"\nDataset Shape: {master_df.shape[0]} rows x {master_df.shape[1]} columns")

print(f"\nRace Coverage:")
print(f"  Total races: {master_df['race_number'].nunique()}")
print(f"  First race: {master_df['race_number'].min()} ({master_df[master_df['race_number']==master_df['race_number'].min()]['race_name'].iloc[0]})")
print(f"  Last race: {master_df['race_number'].max()} ({master_df[master_df['race_number']==master_df['race_number'].max()]['race_name'].iloc[0]})")

print(f"\nSponsor Coverage:")
for sponsor in master_df['primary_sponsor'].unique():
    count = len(master_df[master_df['primary_sponsor'] == sponsor])
    print(f"  {sponsor}: {count} race entries")

print(f"\nExposure Data Summary:")
print(f"  Reddit mentions: {master_df['reddit_mention_count'].sum():,.0f} total")
print(f"  YouTube views: {master_df['youtube_total_view_count'].sum():,.0f} total race views")
print(f"  News mentions: {master_df['news_total_mentions'].sum():,.0f} total")

print(f"\nMissing Values:")
missing = master_df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  None (all missing values filled)")


MASTER DATASET SUMMARY

Dataset Shape: 276 rows x 18 columns

Race Coverage:
  Total races: 53
  First race: 1 (2025 Daytona 500)
  Last race: 53 (2026 Anduril 250)

Sponsor Coverage:
  Busch Light: 53 race entries
  Castrol: 53 race entries
  Love's Travel Stops: 53 race entries
  Other/Unknown: 53 race entries
  Progressive: 52 race entries
  cheddar's Scratch Kitchen: 12 race entries

Exposure Data Summary:
  Reddit mentions: 1,616 total
  YouTube views: 13,966,189 total race views
  News mentions: 48 total

Missing Values:
  None (all missing values filled)


In [21]:
# Save the merged dataset
intermediate_path = 'data/processed/master_dataset_merged.csv'
master_df.to_csv(intermediate_path, index=False)
print(f"\nMerged dataset saved to: {intermediate_path}")

# Save merge statistics
merge_stats = {
    'merge_date': datetime.now().isoformat(),
    'total_rows': len(master_df),
    'unique_races': int(master_df['race_number'].nunique()),
    'unique_sponsors': int(master_df['primary_sponsor'].nunique()),
    'reddit_match_rate': float((master_df['reddit_mention_count'] > 0).mean()),
    'youtube_match_rate': float((master_df['youtube_total_view_count'] > 0).mean()),
    'news_match_rate': float((master_df['news_total_mentions'] > 0).mean())
}

with open('data/processed/merge_statistics.json', 'w') as f:
    json.dump(merge_stats, f, indent=2)

print("Merge statistics saved.")


Merged dataset saved to: data/processed/master_dataset_merged.csv
Merge statistics saved.
